## Exemple concret

Nous allons ici voir un exemple concret afin d'appliquer ce que l'on a vu précédemment.

In [ ]:
from bs4 import BeautifulSoup
import requests

### Exemple Wikipedia

Notre objectif est d'en apprendre plus sur la science des données (data science) grâce à Wikipedia. Nous allons donc **charger la page wikipedia "Science des données"** et **lister tous les articles connexes cités sur cette page**.

In [ ]:
# Nous requêtons, récupérons le contenu et parsons la page souhaitée.
wiki_raw = requests.get("https://fr.wikipedia.org/wiki/Science_des_donn%C3%A9es")
soup = BeautifulSoup(wiki_raw.content)
soup

La `soup` est un peu plus conséquente n'est-ce pas ?

On veut en extraire les **liens vers d'autres articles**, pour pouvoir élargir nos connaissances en data science. On commence par inspecter la page wikipedia dans notre navigateur, ce qui nous apprend que le tag `<main>` contient le corps de l'article. Nous allons restreindre notre analyse à cette partie de la page.

In [ ]:
main_soup = soup.find("main") # Nous n'avons, ici, pas besoin de .find_all() car il n'y a qu'un "main".
main_soup

Les liens sont contenus dans les tags **`a`**. Nous souhaitons donc avoir une **liste** de tous les liens.

In [ ]:
links = main_soup.find_all("a")
links

Certains liens de cette liste de résultats ne renvoient pas vers des articles. On peut remarquer que les `<a>` dont nous avons besoin contiennent tous un attribut `href` (où l'on trouve le texte réel du lien) et un attribut `title` (qui correspond aux liens vers les autres articles). Utilisons cette information pour effectuer une recherche plus précise :

In [ ]:
links = main_soup.find_all("a", href= True, title= True)
links

C'est mieux, mais il reste des éléments qui ne nous intéressent pas. Il faut être encore plus sélectif. On remarque que tous les liens vers des articles commencent par `/wiki/`.

_Remarque : la méthode qui suit est une méthode BONUS._

La méthode **`.startswith()`** en Python vérifie si une chaîne de caractères **commence par une sous-chaîne spécifiée**. Elle retourne `True` si la chaîne commence par la sous-chaîne donnée, sinon `False`. Vous pouvez aussi fournir des indices pour vérifier un sous-ensemble spécifique de la chaîne.

In [ ]:
# Utilisons la méthode startswith
'Bonjour'.startswith("B")

In [ ]:
links = main_soup.find_all("a",
                           href= lambda link: link and link.startswith("/wiki/"), # lambda pour définir une fonction qui prend comme argument link (nous aurions très bien pu mettre caillou ou autre chose)
                                            # link : Si l'attribut href de la balise <a> est présent
                                            # link.startswith("/wiki/") : Si cet attribut commence par la chaîne "/wiki/".
                           title= True)
links

Petit à petit nous avançons ! Extrayons maintenant le texte de chaque lien.

In [ ]:
# Dans un tag, on peut accéder aux attributs avec le slicing habituel.
links[0]['href']  # Dans la liste links, nous accédons au premier élément puis à "href".

In [ ]:
# Pour chaque élément de la liste links nous souhaitons accéder à "href".
list_of_article_links = [ link["href"] for link in links ]
list_of_article_links

Les liens sont-ils corrects ?

Il manque le début de chaque URL ! Après vérification sur Wikipédia, ajoutons à chaque fois "https://fr.wikipedia.org/".

In [ ]:
prefix = "https://fr.wikipedia.org"

_Bonne pratique : essayer sur **1 élément**, puis sur **toute la liste** avec une boucle ou liste en compréhension._

In [ ]:
# Sur 1 lien
prefix + list_of_article_links[0] # Nous effectuons une simple concaténation.

In [ ]:
# La requête HTTP fonctionne désormais.
first_article_content = requests.get(prefix + list_of_article_links[0]).content
first_article_content

Faisons en sorte d'ajouter ce préfix à tous les liens.

In [ ]:
# Pour chaque lien dans list_of_article_links, ajoutons le prefix.
list_article_links_complete = [prefix + extension for extension in list_of_article_links]
list_article_links_complete

On a donc produit un workflow pour obtenir une liste de liens utilisables. Il nous suffirait d'insérer ces étapes dans une **fonction** pour pouvoir ré-effectuer le même processus sur une autre page Wikipedia !

### Bonus : crawling et analyse

Nous pourrions maintenant réaliser quelques analyses sur ces liens. Par exemple, pour chaque article, on veut connaître le **nombre de notes et références**, pour vérifier si wikipedia est bien sourcé...

In [ ]:
from tqdm.notebook import tqdm # Afficher des barres de chargement.
import numpy as np

In [ ]:
# Analyse complète

articles_number_notes = []

for article_link in tqdm(list_article_links_complete[:50]): # tdqm permet d'afficher la barre de progression de la boucle.

    # Requête et parsing
    first_article_content = requests.get(article_link).content
    first_article_soup = BeautifulSoup(first_article_content, 'html.parser')

    # Certains articles n'auront ni notes ni références, ils donneront une erreur.
    # Ces erreurs seront gérées par la commande except.
    try:

        # Nous parcourrons l'arbre pour trouver les éléments souhaités.
        notes_et_references = (first_article_soup
                               .find('main')
                               .find("ol", {"class": "references"})
                               .find_all("li")
                              )
        # Nous stockons dans une liste le nombre de références.
        number_of_notes = len(notes_et_references)
        articles_number_notes.append(number_of_notes)

    except AttributeError:
        print("No note or reference in article", article_link)
        articles_number_notes.append(0) # S'il n'y a pas de notes, nous ajoutons 0 à la liste

print("\nMedian number of notes per article is", np.median(articles_number_notes))
print("Mean number of notes per article is", np.mean(articles_number_notes))
print("Stdev of number of notes per article is", np.std(articles_number_notes))

**L'écart-type est élevé et la moyenne est beaucoup plus haute que la médiane**, ce qui dans notre cas laisse supposer que **certains articles sont très documentés et poussent la moyenne et l'écart-type vers le haut**. Ces articles ne représentent pas la majorité des articles qui ont entre 8 et 9 références (cf. valeur médiane).

On peut présenter ces résultats sous forme de graphiques :

In [ ]:
articles_number_notes

Traçons un boxplot afin d'avoir une visualisation de la répartition des valeurs.

In [ ]:
import matplotlib.pyplot as plt

plt.boxplot(articles_number_notes, patch_artist= True); # patch_artist= True permet de remplir la boîte avec une couleur.

Un autre moyen d'avoir une visualisation de la répartition / distribution des données est d'utiliser le **violin plot**.

In [ ]:
import seaborn as sns
plt.figure(figsize= (4,5))  # Instanciation de la taille de la figure.
sns.violinplot(data= articles_number_notes);